In [15]:
import string
import torch
import re
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [16]:
input_text = "Visvesvaraya National Institute of Technology Nagpur (VNIT), formerly known as Visvesvaraya Regional College of Engineering (VRCE) is a public technical university located in the city of Nagpur, Maharashtra.[4] Established in 1960, the institute is among 31 National Institutes of Technology (NITs) in the country. In 2007, the institute was conferred with the status of Institute of National Importance by the National Institutes of Technology, Science Education and Research Act, 2007 of the Parliament of India with all other NITs. Formerly known as Visvesvaraya Regional College of Engineering (VRCE), the institute is named in honour of an eminent engineer, planner and statesman Sir M. Visvesvaraya. The Institute awards Bachelor's, Master's and Doctorate degrees in engineering, technology, architecture, science and humanities. The institute's history can be traced back to 1947 when the Architecture Department was established by the Madhya Pradesh Government. Following the second five-year plan (1956–60) in India, several industrial projects were contemplated. The Regional Engineering Colleges (RECs) were established by the central government to mimic the IITs at a regional level and act as benchmarks for the other colleges in that state. For the Western region in the year 1960, the institute was established under the name Visvesvaraya Regional College of Engineering (VRCE). It was established under the scheme sponsored by Govt. of India and Govt. of Maharashtra. The college was started in June 1960 by amalgamating the State Govt. Engineering College functioning in Nagpur since July 1956. In the meeting held in October 1962, the governing board of the college resolved to name it after the eminent engineer, planner, and statesman of the country M. Visvesvaraya. The college started functioning in 1960 from a camp office in the premises of Government Polytechnic, Nagpur and subsequently, an area of about 214 acres was acquired to house an independent Regional Engineering College at its present location. It has its jurisdiction over entire state of Maharashtra."
print(input_text)
# Extracting Sentences (tokenizing sentences ig)

sentences = re.split(r'[.!?]', input_text)
print(sentences)

# Preprocessing Each Sentence

preprocessed_sentences = []

for punct in string.punctuation:
  for sentence in sentences:
    sentence = sentence.lower()
    sentence = sentence.replace(punct, "")
    preprocessed_sentences.append(sentence)

print(preprocessed_sentences)

# Creating Vocabulary and tokenizing sentences
vocab = {}
vocab['<UNK>'] = 0

tokenized_sentences = []

for sentence in preprocessed_sentences:

  tokenized_sentence = sentence.split()
  tokenized_sentences.append(tokenized_sentence)

  for word in tokenized_sentence:
    if word not in vocab:
      idx = len(vocab)
      vocab[word] = idx

print(vocab)

# Now we must convert each word in a sentence into it's indices
indexed_sentences = []
for tokenized_sentence in tokenized_sentences:
  indexed_sentence = []
  for word in tokenized_sentence:
    indexed_sentence.append(vocab[word])
  indexed_sentences.append(indexed_sentence)

print(indexed_sentences)

Visvesvaraya National Institute of Technology Nagpur (VNIT), formerly known as Visvesvaraya Regional College of Engineering (VRCE) is a public technical university located in the city of Nagpur, Maharashtra.[4] Established in 1960, the institute is among 31 National Institutes of Technology (NITs) in the country. In 2007, the institute was conferred with the status of Institute of National Importance by the National Institutes of Technology, Science Education and Research Act, 2007 of the Parliament of India with all other NITs. Formerly known as Visvesvaraya Regional College of Engineering (VRCE), the institute is named in honour of an eminent engineer, planner and statesman Sir M. Visvesvaraya. The Institute awards Bachelor's, Master's and Doctorate degrees in engineering, technology, architecture, science and humanities. The institute's history can be traced back to 1947 when the Architecture Department was established by the Madhya Pradesh Government. Following the second five-year

In [17]:
class CustomDatasetSentences(Dataset):

  def __init__(self, indexed_sentences):

    samples = []

    for sentence in indexed_sentences:

      for i in range(1, len(sentence)):
        sampled_input = sentence[:i]
        target = sentence[i]
        samples.append((sampled_input, target))

    self.samples = samples

  def __len__(self):
    return len(self.samples)

  def __getitem__(self, idx): # uss index pe jo bhi sentence hoga wo chahiye
    return torch.tensor(self.samples[idx][0]).view(-1), torch.tensor(self.samples[idx][1])

In [18]:
training_dataset = CustomDatasetSentences(indexed_sentences)
training_dataloader = DataLoader(training_dataset, batch_size=1, shuffle = True)

In [19]:
# Creating RNN Architecture
print(len(vocab))
class RNNnwp(nn.Module):

  def __init__(self, vocab_size):

    super().__init__()
    self.embedding_layer = nn.Embedding(vocab_size, 50)
    self.hidden = nn.RNN(50, 64, batch_first=True) # batch_first = True matlab jab dataloader extract karta h data then uska shape is -> [batch_size, sentence_length]
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, sampled_sentence):

    embedded_vector = self.embedding_layer(sampled_sentence)
    hidden_states, final = self.hidden(embedded_vector) # because final ka shape is (1, batch_size, sentence_length)
    final = final.squeeze(0) # we want only (batch_size, sentence_length) because target is (batch_size) and inn dono format ke shapes rehne pe hi cross_entropy loss calculate karta h.
    pred_output = self.fc(final)

    return pred_output


177


In [20]:
# creating model and initializing hyperparamrters
model = RNNnwp(len(vocab))
model.to(device)

loss_function = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

epochs = 10

In [21]:
for epoch in range(epochs):

  total_loss = 0
  for sample, target in training_dataloader:

    sample = sample.to(device)
    target = target.to(device)

    optimizer.zero_grad()
    # forward pass:

    y_pred = model(sample)

    # loss calculation:

    loss = loss_function(y_pred, target)
    total_loss = total_loss + loss.item()

    # backward pass:

    loss.backward()

    #updation:
    optimizer.step()

  print(f"Epoch: {epoch + 1} | Loss: {total_loss / len(training_dataloader)}")

Epoch: 1 | Loss: 0.898148036810395
Epoch: 2 | Loss: 0.1202589267578944
Epoch: 3 | Loss: 0.10373822223878212
Epoch: 4 | Loss: 0.09523733898466587
Epoch: 5 | Loss: 0.08984854902119001
Epoch: 6 | Loss: 0.08970135327901527
Epoch: 7 | Loss: 0.09405684002668199
Epoch: 8 | Loss: 0.0898859901702848
Epoch: 9 | Loss: 0.08518081999466132
Epoch: 10 | Loss: 0.08178423687100889


In [39]:
test_sentence = "The"
preprocessed_sentence = test_sentence.lower()

for punct in string.punctuation:
    preprocessed_sentence = preprocessed_sentence.replace(punct, "")

tokenized_sentence = preprocessed_sentence.split()

indexed_sentence = []
for word in tokenized_sentence:
    if word in vocab:
        indexed_sentence.append(vocab[word])

for _ in range(11):
    temp = torch.tensor(indexed_sentence).view(1, len(indexed_sentence)).to(device)
    output = model(temp)
    last_output = output.squeeze()
    probs = torch.softmax(last_output, dim=0)
    probs[vocab["<UNK>"]] = 0
    probs = probs / probs.sum()
    index = torch.multinomial(probs, num_samples=1)
    indexed_sentence.append(index.item())

idx_to_word = {i: w for w, i in vocab.items()}

generated_words = [idx_to_word[i] for i in indexed_sentence]

print(" ".join(generated_words))

the college started functioning in 1960 from a camp office in the
